In [0]:
import requests
import pandas as pd
import math
import time
from datetime import datetime, timedelta
import pytz

url = "https://apis.data.go.kr/B552845/risesAndFalls/info"
service_key = dbutils.secrets.get(scope="agrofood-api-prod", key="publicdata-service-key_lyr")




# 한국시간 기준 오늘 날짜
kst = pytz.timezone("Asia/Seoul")
today = datetime.now(kst)
date_str = (today - timedelta(days=3)).strftime("%Y%m%d")

print(f"\n{'='*50}")
print(f"수집 일자: {date_str}")
print(f"{'='*50}")

all_items = []

params = {
    "serviceKey": service_key,
    "returnType": "JSON",
    "pageNo": 1,
    "numOfRows": 1000,
    "cond[exmn_ymd::EQ]": date_str
}

try:
    response = requests.get(url, params=params, timeout=30)
    data = response.json()

    result_code = data.get("response", {}).get("header", {}).get("resultCode")
    result_msg = data.get("response", {}).get("header", {}).get("resultMsg")
    print(f"응답코드: {result_code} / 응답메시지: {result_msg}")

    total_count = int(data.get("response", {}).get("body", {}).get("totalCount", 0))
    num_of_rows = int(data.get("response", {}).get("body", {}).get("numOfRows", 1000))
    pages = math.ceil(total_count / num_of_rows) if total_count > 0 else 0

    print(f"총 건수: {total_count} / 총 페이지 수: {pages}")

    if total_count == 0:
        print("데이터 없음")
    else:
        collect_time = datetime.now(pytz.timezone("Asia/Seoul")).strftime("%Y-%m-%d_%H:%M:%S")

        for page in range(1, pages + 1):
            params["pageNo"] = page

            response = requests.get(url, params=params, timeout=30)
            data = response.json()

            items = data.get("response", {}).get("body", {}).get("items", {}).get("item", [])

            if isinstance(items, dict):
                items = [items]

            if not items:
                print(f"[{page}/{pages} 페이지] 데이터 없음")
                continue

            for item in items:
                item["collect_time"] = collect_time

            all_items.extend(items)
            print(f"  [{page}/{pages} 페이지] 수집 건수: {len(items)} / 누적: {len(all_items)}")

            time.sleep(0.3)

except Exception as e:
    print(f"에러 발생: {e}")

df = pd.DataFrame(all_items)
file_name = f"/Volumes/bronze_api/agrofood_risesandfalls/volumn/가격등락정보_{date_str}.csv"
df.to_csv(file_name, index=False, encoding="utf-8-sig")

spark_df = spark.read.csv(file_name, header=True, inferSchema=False)

spark_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze_api.agrofood_risesandfalls.risesandfalls")


print(f"\n{'='*50}")
print("저장 완료")
print(f"총 행 수: {len(df)}")
print(f"총 열 수: {len(df.columns)}")
print(f"저장 파일명: {file_name}")
print(df.head())

In [0]:
# spark_df.printSchema()
# spark.table("bronze_api.agrofood_risesandfalls.risesandfalls").printSchema()

In [0]:
# spark.sql("DESCRIBE TABLE bronze_api.agrofood_risesandfalls.risesandfalls").show(200, truncate=False)